In [ ]:
# =============================================================================
# STEP 1: ENVIRONMENT SETUP
# PixelNet - Road Scene Semantic Segmentation
# =============================================================================
# Run this cell first in your Jupyter Notebook or as a standalone script.
# This installs all required libraries and imports all dependencies.

# -----------------------------------------------------------------------------
# 1A. Install Required Libraries
# -----------------------------------------------------------------------------
# Uncomment and run the lines below if running for the first time:
#
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install tqdm matplotlib numpy Pillow kaggle scikit-learn opencv-python

# -----------------------------------------------------------------------------
# 1B. Standard Library Imports
# -----------------------------------------------------------------------------
import os
import sys
import json
import shutil
import zipfile
import random
import warnings
warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# 1C. Numerical & Image Processing Imports
# -----------------------------------------------------------------------------
import numpy as np
from PIL import Image
import cv2                          # OpenCV for image I/O and resizing

# -----------------------------------------------------------------------------
# 1D. PyTorch Core Imports
# -----------------------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# -----------------------------------------------------------------------------
# 1E. TorchVision Imports (Backbone + Transforms)
# -----------------------------------------------------------------------------
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

# -----------------------------------------------------------------------------
# 1F. Visualization Imports
# -----------------------------------------------------------------------------
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

# -----------------------------------------------------------------------------
# 1G. Progress Bar & Utilities
# -----------------------------------------------------------------------------
from tqdm import tqdm                       # Epoch/batch progress bars
from sklearn.model_selection import train_test_split

# -----------------------------------------------------------------------------
# 1H. Device Configuration
# -----------------------------------------------------------------------------
# Automatically use GPU if available, otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

if torch.cuda.is_available():
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")
    print(f"[INFO] CUDA Version: {torch.version.cuda}")
else:
    print("[INFO] Running on CPU — training will be slower.")

# -----------------------------------------------------------------------------
# 1I. Global Hyperparameters & Configuration
# -----------------------------------------------------------------------------
CONFIG = {
    # --- Dataset ---
    "dataset_name"      : "road_segmentation",  # Name tag for folder structure
    "image_size"        : (256, 256),            # (H, W) all images resized to this
    "num_classes"       : 8,                     # Number of segmentation classes
    "ignore_index"      : 255,                   # Pixels to ignore in loss (void)

    # --- PixelNet Sampling ---
    "num_samples_train" : 2048,                  # Pixels sampled per image (training)
    "num_samples_val"   : 4096,                  # Pixels sampled per image (validation)

    # --- Training ---
    "batch_size"        : 4,                     # Images per batch
    "num_epochs"        : 30,                    # Total training epochs
    "learning_rate"     : 1e-4,                  # Adam optimizer LR
    "weight_decay"      : 5e-4,                  # L2 regularization
    "val_split"         : 0.2,                   # 20% images reserved for validation

    # --- Paths ---
    "data_root"         : "./data",              # Root directory for dataset
    "output_dir"        : "./outputs",           # Where models/logs/visuals are saved
    "checkpoint_dir"    : "./checkpoints",       # Saved model weights

    # --- Reproducibility ---
    "seed"              : 42,
}

# -----------------------------------------------------------------------------
# 1J. Seed Everything for Reproducibility
# -----------------------------------------------------------------------------
def set_seed(seed: int):
    """Fix all random seeds for reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
print(f"[INFO] Random seed set to: {CONFIG['seed']}")

# -----------------------------------------------------------------------------
# 1K. Create Output Directories
# -----------------------------------------------------------------------------
for dir_path in [CONFIG["data_root"], CONFIG["output_dir"], CONFIG["checkpoint_dir"]]:
    os.makedirs(dir_path, exist_ok=True)

print(f"[INFO] Output directories created:")
print(f"       Data     → {CONFIG['data_root']}")
print(f"       Outputs  → {CONFIG['output_dir']}")
print(f"       Checkpts → {CONFIG['checkpoint_dir']}")

# -----------------------------------------------------------------------------
# 1L. Cityscapes-style Class Definitions (8 Merged Classes)
# -----------------------------------------------------------------------------
# These are simplified/merged Cityscapes classes for road scene segmentation.
# Mapping fine labels (0–33) → coarse labels (0–7).

CLASS_NAMES = [
    "flat",           # 0 — road, sidewalk, parking
    "construction",   # 1 — building, wall, fence, guard rail
    "object",         # 2 — pole, traffic light, traffic sign
    "nature",         # 3 — vegetation, terrain
    "sky",            # 4 — sky
    "human",          # 5 — person, rider
    "vehicle",        # 6 — car, truck, bus, train, motorcycle, bicycle
    "void",           # 7 — unlabeled, ego vehicle, rectification border
]

# Distinct color palette for visualization (R, G, B)
CLASS_COLORS = np.array([
    [128, 64, 128],    # flat        — purple-ish road color
    [ 70, 70,  70],    # construction — dark grey
    [153, 153, 153],   # object       — grey
    [107, 142,  35],   # nature       — olive green
    [ 70, 130, 180],   # sky          — steel blue
    [220,  20,  60],   # human        — crimson
    [  0,  0, 142],    # vehicle      — dark blue
    [  0,  0,   0],    # void         — black
], dtype=np.uint8)

print(f"\n[INFO] Classes ({CONFIG['num_classes']} total):")
for i, name in enumerate(CLASS_NAMES):
    print(f"       [{i}] {name}")

print("\n[INFO] STEP 1 COMPLETE — Environment ready.")
print("=" * 60)

[INFO] Using device: cpu
[INFO] Running on CPU — training will be slower.
[INFO] Random seed set to: 42
[INFO] Output directories created:
       Data     → ./data
       Outputs  → ./outputs
       Checkpts → ./checkpoints

[INFO] Classes (8 total):
       [0] flat
       [1] construction
       [2] object
       [3] nature
       [4] sky
       [5] human
       [6] vehicle
       [7] void

[INFO] STEP 1 COMPLETE — Environment ready.


In [4]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"soumojaykarmakar","key":"b2ac274459ff645ba55d3fe7d026fa8a"}'}

In [5]:
import os
import shutil

# Create kaggle directory
os.makedirs("/root/.kaggle", exist_ok=True)

# Move file
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")

# Set permissions (IMPORTANT)
os.chmod("/root/.kaggle/kaggle.json", 600)

In [6]:
!pip install kaggle

In [7]:
!kaggle datasets list

ref                                                          title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
nalisha/job-salary-prediction-dataset                        Job Salary Prediction Dataset                          3144815  2026-03-16 19:54:33.843000           3631         81                1  
ssssws/chocolate-sales-dataset-2023-2024                     Chocolate Sales Dataset 2023 - 2024                   24420255  2026-03-07 04:58:02.387000           7043        116                1  
sharmajicoder/student-mental-health-and-burnout              Student Mental Health and Burnout                    137596875  2026-03-18 19:03:34.140000           1187         31                1  
grandmaster07/s

In [10]:
# =============================================================================
# STEP 2: DATASET DOWNLOAD & SETUP
# PixelNet - Road Scene Semantic Segmentation
# =============================================================================
# This step handles:
#   - Loading Kaggle API credentials from kaggle.json
#   - Downloading a road segmentation dataset from Kaggle
#   - Extracting and organizing into images/ and masks/ folders
#   - Generating a dataset summary report
#
# SUPPORTED DATASET (default):
#   Kaggle: "dansbecker/cityscapes-image-pairs"  (Cityscapes-style road scenes)
#   Alternatively: any semantic segmentation dataset with image+mask pairs
# =============================================================================

import os
import sys
import json
import shutil
import zipfile
import random
import warnings
import numpy as np
from PIL import Image
from tqdm import tqdm
warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# Re-import CONFIG (copy from Step 1, or import if using a shared config.py)
# -----------------------------------------------------------------------------
CONFIG = {
    "dataset_name"      : "road_segmentation",
    "image_size"        : (256, 256),
    "num_classes"       : 8,
    "ignore_index"      : 255,
    "num_samples_train" : 2048,
    "num_samples_val"   : 4096,
    "batch_size"        : 4,
    "num_epochs"        : 30,
    "learning_rate"     : 1e-4,
    "weight_decay"      : 5e-4,
    "val_split"         : 0.2,
    "data_root"         : "./data",
    "output_dir"        : "./outputs",
    "checkpoint_dir"    : "./checkpoints",
    "seed"              : 42,
}

# Cityscapes RGB color palette → 8 merged coarse class indices
COARSE_COLOR_TO_CLASS = {
    (128,  64, 128): 0,   # flat        (road, sidewalk)
    ( 70,  70,  70): 1,   # construction (building, wall, fence)
    (153, 153, 153): 2,   # object       (pole, sign, light)
    (107, 142,  35): 3,   # nature       (vegetation, terrain)
    ( 70, 130, 180): 4,   # sky
    (220,  20,  60): 5,   # human        (person, rider)
    (  0,   0, 142): 6,   # vehicle      (car, truck, bus, etc.)
    (  0,   0,   0): 7,   # void         (unlabeled, ego vehicle)
}

CLASS_NAMES = [
    "flat", "construction", "object",
    "nature", "sky", "human", "vehicle", "void"
]

# =============================================================================
# 2A. KAGGLE AUTHENTICATION
# =============================================================================

def setup_kaggle_credentials(kaggle_json_path: str = "kaggle.json") -> bool:
    """
    Load and install Kaggle API credentials from a kaggle.json file.

    The kaggle.json file format:
        {"username": "your_kaggle_username", "key": "your_api_key"}

    Download it from: https://www.kaggle.com/settings → API → Create New Token

    Args:
        kaggle_json_path: Path to your kaggle.json credentials file.

    Returns:
        True if credentials installed successfully, False otherwise.
    """
    print("[2A] Setting up Kaggle credentials...")

    if not os.path.exists(kaggle_json_path):
        print(f"[ERROR] kaggle.json not found at '{kaggle_json_path}'")
        print("        Download from: https://www.kaggle.com/settings → API → Create Token")
        return False

    with open(kaggle_json_path, "r") as f:
        creds = json.load(f)

    if not {"username", "key"}.issubset(creds.keys()):
        print("[ERROR] kaggle.json must contain 'username' and 'key' fields.")
        return False

    # Install to ~/.kaggle/ (standard Kaggle CLI location)
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    dest = os.path.join(kaggle_dir, "kaggle.json")
    if os.path.abspath(kaggle_json_path) != os.path.abspath(dest):
      shutil.copy(kaggle_json_path, dest)
    else:
      print("[INFO] kaggle.json already in correct location. Skipping copy.")

    try:
        os.chmod(dest, 0o600)   # Required permission on Linux/macOS
    except Exception:
        pass                    # Windows: skip chmod

    print(f"[2A] Credentials installed → {dest}")
    print(f"[2A] Logged in as: {creds['username']}")
    return True


# =============================================================================
# 2B. DATASET DOWNLOAD FROM KAGGLE
# =============================================================================

def download_dataset_from_kaggle(
    dataset_slug: str,
    download_dir: str,
    kaggle_json_path: str = "kaggle.json"
) -> str:
    """
    Download a Kaggle dataset using the Kaggle CLI tool.

    Args:
        dataset_slug     : Kaggle dataset ID, e.g. "dansbecker/cityscapes-image-pairs"
        download_dir     : Local folder to save the downloaded zip.
        kaggle_json_path : Path to kaggle.json API credentials.

    Returns:
        Full path to the downloaded .zip file.
    """
    print(f"\n[2B] Downloading Kaggle dataset: {dataset_slug}")

    if not setup_kaggle_credentials(kaggle_json_path):
        raise FileNotFoundError("Kaggle credentials failed. Cannot download.")

    os.makedirs(download_dir, exist_ok=True)

    # Check if already downloaded
    zip_name = dataset_slug.split("/")[-1] + ".zip"
    zip_path = os.path.join(download_dir, zip_name)
    if os.path.exists(zip_path):
        print(f"[2B] Already downloaded: {zip_path}")
        return zip_path

    # Run Kaggle CLI
    cmd = f"kaggle datasets download -d {dataset_slug} -p {download_dir} --quiet"
    print(f"[2B] Command: {cmd}")
    ret = os.system(cmd)

    if ret != 0:
        raise RuntimeError(
            f"Kaggle download failed (exit code {ret}).\n"
            "Tips:\n"
            "  1. Run: pip install kaggle\n"
            "  2. Verify your API key at https://www.kaggle.com/settings\n"
            "  3. Accept dataset terms on Kaggle website if required."
        )

    for fname in os.listdir(download_dir):
        if fname.endswith(".zip"):
            zip_path = os.path.join(download_dir, fname)
            print(f"[2B] Saved to: {zip_path}")
            return zip_path

    raise FileNotFoundError("No .zip found after download.")


# =============================================================================
# 2C. RGB MASK → SINGLE-CHANNEL CLASS INDEX MASK
# =============================================================================

def rgb_mask_to_class_index(mask_rgb: np.ndarray) -> np.ndarray:
    """
    Convert an RGB-colored segmentation mask into a single-channel
    class-index mask where each pixel value = class ID (0–7).

    Unknown colors are assigned to class 7 (void).

    Args:
        mask_rgb: numpy array of shape (H, W, 3), dtype=uint8.

    Returns:
        numpy array of shape (H, W), dtype=uint8, values in [0, 7].
    """
    H, W = mask_rgb.shape[:2]
    # Default everything to void (class 7)
    class_mask = np.full((H, W), fill_value=7, dtype=np.uint8)

    for (r, g, b), cls_idx in COARSE_COLOR_TO_CLASS.items():
        ref = np.array([r, g, b], dtype=np.int16)
        # Allow ±15 tolerance to handle JPEG compression artifacts
        diff = np.abs(mask_rgb.astype(np.int16) - ref)
        matched = np.all(diff <= 15, axis=-1)
        class_mask[matched] = cls_idx

    return class_mask


# =============================================================================
# 2D. EXTRACT & ORGANIZE INTO train/ AND val/ FOLDERS
# =============================================================================

def extract_and_organize(zip_path: str, data_root: str) -> dict:
    """
    Extract zip and organize files into structured train/val directories.

    Directory layout created:
        data/
        ├── train/
        │   ├── images/     ← RGB input images
        │   └── masks/      ← Class-index segmentation masks
        └── val/
            ├── images/
            └── masks/

    For Cityscapes-pairs format: each raw file is a side-by-side
    concatenation [RGB photo | colored mask]. This function splits them.

    Args:
        zip_path  : Path to downloaded .zip file.
        data_root : Root directory for organized dataset output.

    Returns:
        dataset_info dict mapping split → {"images": [...], "masks": [...]}
    """
    print(f"\n[2D] Extracting: {zip_path}")

    extract_dir = os.path.join(data_root, "_raw_extracted")
    os.makedirs(extract_dir, exist_ok=True)

    # --- Extract only if not already done ---
    if not any(os.scandir(extract_dir)):
        with zipfile.ZipFile(zip_path, "r") as zf:
            members = zf.namelist()
            for member in tqdm(members, desc="  Unzipping", unit="file"):
                zf.extract(member, extract_dir)
        print(f"[2D] Extracted {len(members)} files → {extract_dir}")
    else:
        print(f"[2D] Already extracted, skipping.")

    # Create output directories
    for split in ["train", "val"]:
        for sub in ["images", "masks"]:
            os.makedirs(os.path.join(data_root, split, sub), exist_ok=True)

    # Collect all image files from extracted folder
    all_files = []
    for root, _, files in os.walk(extract_dir):
        for fname in sorted(files):
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                all_files.append(os.path.join(root, fname))

    print(f"[2D] Total raw files found: {len(all_files)}")

    # --- Train / Val split (80 / 20) ---
    random.seed(42)
    random.shuffle(all_files)
    n_val       = max(1, int(len(all_files) * CONFIG["val_split"]))
    split_map   = {
        "val"  : all_files[:n_val],
        "train": all_files[n_val:]
    }

    dataset_info = {
        "train": {"images": [], "masks": []},
        "val"  : {"images": [], "masks": []}
    }

    # --- Process and save ---
    for split, file_list in split_map.items():
        print(f"\n[2D] Organizing {split} split ({len(file_list)} files)...")

        for idx, fpath in enumerate(tqdm(file_list, desc=f"  {split}", unit="img")):
            img_pil = Image.open(fpath).convert("RGB")
            W, H    = img_pil.size

            # --- Detect side-by-side Cityscapes-pairs format ---
            if W > H * 1.5:
                # Left half = photo, Right half = colored semantic mask
                half_w = W // 2
                photo  = img_pil.crop((0,      0, half_w, H))
                mask_c = img_pil.crop((half_w, 0, W,      H))
            else:
                # Standalone image without paired mask (generate void mask)
                photo  = img_pil
                mask_c = Image.fromarray(np.zeros((H, W, 3), dtype=np.uint8))

            # Convert RGB mask → class-index mask
            mask_idx = rgb_mask_to_class_index(np.array(mask_c))
            mask_pil = Image.fromarray(mask_idx)

            # Save to organized directories
            base          = f"{split}_{idx:05d}.png"
            img_out_path  = os.path.join(data_root, split, "images", base)
            mask_out_path = os.path.join(data_root, split, "masks",  base)

            photo.save(img_out_path)
            mask_pil.save(mask_out_path)

            dataset_info[split]["images"].append(img_out_path)
            dataset_info[split]["masks"].append(mask_out_path)

    return dataset_info


# =============================================================================
# 2E. ALTERNATIVE: LOAD FROM LOCAL DIRECTORIES
# =============================================================================

def load_from_local_directory(
    images_dir: str,
    masks_dir: str,
    data_root: str,
    val_split: float = 0.2
) -> dict:
    """
    Use this when you already have images and masks in two flat directories.
    File names must match between images_dir and masks_dir.

    Directory structure expected:
        images_dir/   img_001.png, img_002.png ...
        masks_dir/    img_001.png, img_002.png ...   ← same names

    Args:
        images_dir : Folder containing RGB images.
        masks_dir  : Folder containing single-channel or RGB mask images.
        data_root  : Destination root for organized train/val split.
        val_split  : Fraction for validation set.

    Returns:
        dataset_info dict.
    """
    print(f"\n[2E] Loading from local directories...")
    print(f"     Images : {images_dir}")
    print(f"     Masks  : {masks_dir}")

    exts = (".png", ".jpg", ".jpeg")
    img_files  = sorted(f for f in os.listdir(images_dir) if f.lower().endswith(exts))
    mask_files = sorted(f for f in os.listdir(masks_dir)  if f.lower().endswith(exts))

    assert len(img_files) == len(mask_files), (
        f"Count mismatch: {len(img_files)} images vs {len(mask_files)} masks"
    )
    print(f"[2E] Found {len(img_files)} matched pairs.")

    # Train/Val split
    all_idx = list(range(len(img_files)))
    random.seed(42)
    random.shuffle(all_idx)
    n_val     = max(1, int(len(all_idx) * val_split))
    split_map = {"val": all_idx[:n_val], "train": all_idx[n_val:]}

    dataset_info = {
        "train": {"images": [], "masks": []},
        "val"  : {"images": [], "masks": []}
    }

    for split, idx_list in split_map.items():
        os.makedirs(os.path.join(data_root, split, "images"), exist_ok=True)
        os.makedirs(os.path.join(data_root, split, "masks"),  exist_ok=True)

        for i, idx in enumerate(tqdm(idx_list, desc=f"  Organizing {split}")):
            src_img  = os.path.join(images_dir, img_files[idx])
            src_mask = os.path.join(masks_dir,  mask_files[idx])
            dst_img  = os.path.join(data_root, split, "images", f"{split}_{i:05d}.png")
            dst_mask = os.path.join(data_root, split, "masks",  f"{split}_{i:05d}.png")
            shutil.copy(src_img,  dst_img)
            shutil.copy(src_mask, dst_mask)
            dataset_info[split]["images"].append(dst_img)
            dataset_info[split]["masks"].append(dst_mask)

    return dataset_info


# =============================================================================
# 2F. DATASET SUMMARY REPORT
# =============================================================================

def print_dataset_summary(dataset_info: dict, data_root: str):
    """Print a formatted summary of the organized dataset."""
    print("\n" + "=" * 60)
    print("  DATASET SUMMARY REPORT")
    print("=" * 60)
    total = 0
    for split, info in dataset_info.items():
        n = len(info["images"])
        total += n
        print(f"  {split.upper():6s} → {n:5d} image-mask pairs")
        print(f"          images/ : {os.path.join(data_root, split, 'images')}")
        print(f"          masks/  : {os.path.join(data_root, split, 'masks')}")
    print(f"\n  TOTAL  → {total:5d} samples")
    print("-" * 60)

    # Verify one sample
    if dataset_info["train"]["images"]:
        sp_img  = np.array(Image.open(dataset_info["train"]["images"][0]))
        sp_mask = np.array(Image.open(dataset_info["train"]["masks"][0]))
        print(f"  Sample Image  → shape: {sp_img.shape}, dtype: {sp_img.dtype}")
        print(f"  Sample Mask   → shape: {sp_mask.shape}, dtype: {sp_mask.dtype}")
        print(f"  Unique class IDs in mask: {np.unique(sp_mask).tolist()}")
    print("=" * 60)


# =============================================================================
# 2G. MASTER SETUP FUNCTION
# =============================================================================

def setup_dataset(
    method           : str = "kaggle",
    kaggle_json_path : str = "kaggle.json",
    kaggle_dataset   : str = "dansbecker/cityscapes-image-pairs",
    local_images_dir : str = None,
    local_masks_dir  : str = None,
) -> dict:
    """
    Master dataset setup function. Supports two methods:

        method="kaggle" → Download from Kaggle API (requires kaggle.json)
        method="local"  → Use existing local images + masks directories

    Args:
        method           : "kaggle" or "local"
        kaggle_json_path : Path to kaggle.json credentials file.
        kaggle_dataset   : Kaggle dataset slug (for method="kaggle").
        local_images_dir : Path to images folder (for method="local").
        local_masks_dir  : Path to masks folder  (for method="local").

    Returns:
        dataset_info dict: {"train": {"images": [...], "masks": [...]}, "val": {...}}
    """
    data_root     = CONFIG["data_root"]
    train_img_dir = os.path.join(data_root, "train", "images")

    # --- Reuse existing organized dataset if present ---
    if os.path.exists(train_img_dir) and len(os.listdir(train_img_dir)) > 0:
        print("[2G] Organized dataset already exists. Loading paths...")
        dataset_info = {"train": {"images": [], "masks": []},
                        "val"  : {"images": [], "masks": []}}
        for split in ["train", "val"]:
            img_dir  = os.path.join(data_root, split, "images")
            mask_dir = os.path.join(data_root, split, "masks")
            if os.path.exists(img_dir):
                imgs  = sorted(os.path.join(img_dir, f)
                               for f in os.listdir(img_dir) if f.endswith(".png"))
                masks = sorted(os.path.join(mask_dir, f)
                               for f in os.listdir(mask_dir) if f.endswith(".png"))
                dataset_info[split]["images"] = imgs
                dataset_info[split]["masks"]  = masks
        print_dataset_summary(dataset_info, data_root)
        return dataset_info

    # --- Download + organize ---
    if method == "kaggle":
        zip_path     = download_dataset_from_kaggle(
            dataset_slug     = kaggle_dataset,
            download_dir     = os.path.join(data_root, "_downloads"),
            kaggle_json_path = kaggle_json_path,
        )
        dataset_info = extract_and_organize(zip_path, data_root)

    elif method == "local":
        assert local_images_dir and local_masks_dir, \
            "Provide local_images_dir and local_masks_dir when using method='local'."
        dataset_info = load_from_local_directory(
            images_dir = local_images_dir,
            masks_dir  = local_masks_dir,
            data_root  = data_root,
            val_split  = CONFIG["val_split"],
        )
    else:
        raise ValueError(f"Unknown method='{method}'. Use 'kaggle' or 'local'.")

    print_dataset_summary(dataset_info, data_root)
    return dataset_info


# =============================================================================
# USAGE EXAMPLES
# =============================================================================
#
#   ┌─────────────────────────────────────────────────────────────────┐
#   │  OPTION A: Download from Kaggle                                 │
#   │                                                                 │
#   │  dataset_info = setup_dataset(                                  │
#   │      method           = "kaggle",                               │
#   │      kaggle_json_path = "kaggle.json",   # ← your API key file │
#   │      kaggle_dataset   = "dansbecker/cityscapes-image-pairs",    │
#   │  )                                                              │
#   ├─────────────────────────────────────────────────────────────────┤
#   │  OPTION B: Use local data                                       │
#   │                                                                 │
#   │  dataset_info = setup_dataset(                                  │
#   │      method           = "local",                                │
#   │      local_images_dir = "/path/to/images",                      │
#   │      local_masks_dir  = "/path/to/masks",                       │
#   │  )                                                              │
#   └─────────────────────────────────────────────────────────────────┘

if __name__ == "__main__":
      dataset_info = setup_dataset(
      method="kaggle",
      kaggle_json_path="/root/.kaggle/kaggle.json",
      kaggle_dataset="dansbecker/cityscapes-image-pairs"
  )
print("\n[INFO] STEP 2 COMPLETE — Dataset organized and ready.")
print("=" * 60)


[2B] Downloading Kaggle dataset: dansbecker/cityscapes-image-pairs
[2A] Setting up Kaggle credentials...
[INFO] kaggle.json already in correct location. Skipping copy.
[2A] Credentials installed → /root/.kaggle/kaggle.json
[2A] Logged in as: soumojaykarmakar
[2B] Command: kaggle datasets download -d dansbecker/cityscapes-image-pairs -p ./data/_downloads --quiet
[2B] Saved to: ./data/_downloads/cityscapes-image-pairs.zip

[2D] Extracting: ./data/_downloads/cityscapes-image-pairs.zip


  Unzipping: 100%|██████████| 6950/6950 [00:04<00:00, 1596.73file/s]


[2D] Extracted 6950 files → ./data/_raw_extracted
[2D] Total raw files found: 6950

[2D] Organizing val split (1390 files)...


  val: 100%|██████████| 1390/1390 [00:58<00:00, 23.56img/s]



[2D] Organizing train split (5560 files)...


  train: 100%|██████████| 5560/5560 [03:58<00:00, 23.31img/s]



  DATASET SUMMARY REPORT
  TRAIN  →  5560 image-mask pairs
          images/ : ./data/train/images
          masks/  : ./data/train/masks
  VAL    →  1390 image-mask pairs
          images/ : ./data/val/images
          masks/  : ./data/val/masks

  TOTAL  →  6950 samples
------------------------------------------------------------
  Sample Image  → shape: (256, 256, 3), dtype: uint8
  Sample Mask   → shape: (256, 256), dtype: uint8
  Unique class IDs in mask: [0, 1, 2, 3, 4, 5, 6, 7]

[INFO] STEP 2 COMPLETE — Dataset organized and ready.


In [11]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
import torchvision.transforms as transforms

class RoadSegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, image_size=(256, 256)):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.image_size = image_size

        # Image transforms
        self.image_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5],
                                 std=[0.5, 0.5, 0.5])
        ])

        # Mask transform (NO normalization)
        self.mask_transform = transforms.Compose([
            transforms.Resize(image_size, interpolation=Image.NEAREST)
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image
        image = Image.open(self.image_paths[idx]).convert("RGB")
        image = self.image_transform(image)

        # Load mask
        mask = Image.open(self.mask_paths[idx])
        mask = self.mask_transform(mask)

        # Convert mask to tensor (H, W)
        mask = torch.from_numpy(np.array(mask)).long()

        return image, mask

In [12]:
from torch.utils.data import DataLoader

train_dataset = RoadSegmentationDataset(
    dataset_info["train"]["images"],
    dataset_info["train"]["masks"]
)

val_dataset = RoadSegmentationDataset(
    dataset_info["val"]["images"],
    dataset_info["val"]["masks"]
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

Train samples: 5560
Val samples: 1390


In [13]:
import torchvision.models as models
import torch.nn as nn

class VGG16_Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super(VGG16_Backbone, self).__init__()

        vgg = models.vgg16(pretrained=pretrained).features

        # Split VGG into stages (important for hypercolumns)
        self.stage1 = vgg[:5]    # conv1_1 to conv1_2
        self.stage2 = vgg[5:10]  # conv2
        self.stage3 = vgg[10:17] # conv3
        self.stage4 = vgg[17:24] # conv4
        self.stage5 = vgg[24:31] # conv5

    def forward(self, x):
        f1 = self.stage1(x)  # low-level features
        f2 = self.stage2(f1)
        f3 = self.stage3(f2)
        f4 = self.stage4(f3)
        f5 = self.stage5(f4)  # high-level features

        return [f1, f2, f3, f4, f5]

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

backbone = VGG16_Backbone().to(device)

x = torch.randn(1, 3, 256, 256).to(device)
features = backbone(x)

for i, f in enumerate(features):
    print(f"Feature {i+1} shape:", f.shape)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:05<00:00, 99.9MB/s]


Feature 1 shape: torch.Size([1, 64, 128, 128])
Feature 2 shape: torch.Size([1, 128, 64, 64])
Feature 3 shape: torch.Size([1, 256, 32, 32])
Feature 4 shape: torch.Size([1, 512, 16, 16])
Feature 5 shape: torch.Size([1, 512, 8, 8])


In [15]:
import torch.nn.functional as F

class Hypercolumn(nn.Module):
    def __init__(self):
        super(Hypercolumn, self).__init__()

    def forward(self, features):
        """
        features: list of feature maps [f1, f2, f3, f4, f5]
        """

        # Target size = size of first feature map (highest resolution)
        target_size = features[0].shape[2:]  # (H, W)

        upsampled_features = []

        for f in features:
            # Upsample each feature map to target size
            upsampled = F.interpolate(
                f,
                size=target_size,
                mode='bilinear',
                align_corners=False
            )
            upsampled_features.append(upsampled)

        # 🔥 Concatenate along channel dimension
        hypercolumn = torch.cat(upsampled_features, dim=1)

        return hypercolumn

In [16]:
hyper = Hypercolumn().to(device)

features = backbone(x)
hc = hyper(features)

print("Hypercolumn shape:", hc.shape)

Hypercolumn shape: torch.Size([1, 1472, 128, 128])


In [17]:
def sample_pixels(hypercolumn, mask, num_samples=2048):
    """
    hypercolumn: [B, C, H, W]
    mask: [B, H, W]

    Returns:
        sampled_features: [N, C]
        sampled_labels: [N]
    """

    B, C, H, W = hypercolumn.shape

    # Flatten spatial dimensions
    hypercolumn = hypercolumn.permute(0, 2, 3, 1).reshape(-1, C)  # [B*H*W, C]
    mask = mask.view(-1)  # [B*H*W]

    # Random sampling indices
    total_pixels = hypercolumn.shape[0]
    indices = torch.randperm(total_pixels)[:num_samples]

    sampled_features = hypercolumn[indices]
    sampled_labels = mask[indices]

    return sampled_features, sampled_labels

In [18]:
hc = hyper(features)           # [B, C, H, W]
images, masks = next(iter(train_loader))

images = images.to(device)
masks = masks.to(device)

features = backbone(images)
hc = hyper(features)

sampled_feats, sampled_labels = sample_pixels(hc, masks)

print("Sampled features:", sampled_feats.shape)
print("Sampled labels:", sampled_labels.shape)

Sampled features: torch.Size([2048, 1472])
Sampled labels: torch.Size([2048])


In [19]:
class PixelClassifier(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(PixelClassifier, self).__init__()

        self.mlp = nn.Sequential(
            nn.Linear(in_channels, 1024),
            nn.ReLU(inplace=True),

            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),

            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        """
        x: [N, C] → sampled pixel features
        """
        return self.mlp(x)

In [20]:
# Hypercolumn channels (from STEP 5)
hypercolumn_channels = 1472  # 64+128+256+512+512

num_classes = 8  # from your dataset

classifier = PixelClassifier(
    in_channels=hypercolumn_channels,
    num_classes=num_classes
).to(device)

In [21]:
# Sample input
x = torch.randn(2048, hypercolumn_channels).to(device)

out = classifier(x)
print("Output shape:", out.shape)

Output shape: torch.Size([2048, 8])


In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    list(backbone.parameters()) + list(classifier.parameters()),
    lr=1e-4
)

num_epochs = 10

for epoch in range(num_epochs):
    backbone.train()
    classifier.train()

    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)

    for images, masks in loop:
        images = images.to(device)
        masks = masks.to(device)

        # -------------------------------
        # 1. Forward pass (Backbone)
        # -------------------------------
        features = backbone(images)

        # -------------------------------
        # 2. Hypercolumn
        # -------------------------------
        hc = hyper(features)

        # -------------------------------
        # 3. Pixel Sampling
        # -------------------------------
        sampled_feats, sampled_labels = sample_pixels(hc, masks, num_samples=2048)

        sampled_feats = sampled_feats.to(device)
        sampled_labels = sampled_labels.to(device)

        # -------------------------------
        # 4. MLP Prediction
        # -------------------------------
        outputs = classifier(sampled_feats)

        # -------------------------------
        # 5. Loss
        # -------------------------------
        loss = criterion(outputs, sampled_labels)

        # -------------------------------
        # 6. Backprop
        # -------------------------------
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}")

Epoch [1/10]:  18%|█▊        | 253/1390 [59:59<4:27:30, 14.12s/it, loss=1.63]

In [ ]:
class DensePredictor(nn.Module):
    def __init__(self, classifier, in_channels, num_classes):
        super(DensePredictor, self).__init__()

        # Convert MLP to Conv layers
        self.conv1 = nn.Conv2d(in_channels, 1024, kernel_size=1)
        self.relu1 = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(1024, 512, kernel_size=1)
        self.relu2 = nn.ReLU(inplace=True)

        self.conv3 = nn.Conv2d(512, num_classes, kernel_size=1)

        # Copy weights from trained MLP
        self._load_from_mlp(classifier)

    def _load_from_mlp(self, classifier):
        self.conv1.weight.data = classifier.mlp[0].weight.data.view(1024, -1, 1, 1)
        self.conv1.bias.data   = classifier.mlp[0].bias.data

        self.conv2.weight.data = classifier.mlp[2].weight.data.view(512, -1, 1, 1)
        self.conv2.bias.data   = classifier.mlp[2].bias.data

        self.conv3.weight.data = classifier.mlp[4].weight.data.view(-1, 512, 1, 1)
        self.conv3.bias.data   = classifier.mlp[4].bias.data

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.conv3(x)
        return x

In [ ]:
dense_predictor = DensePredictor(
    classifier,
    in_channels=1472,
    num_classes=8
).to(device)

In [ ]:
def predict_full_image(image, backbone, hyper, dense_predictor):
    backbone.eval()
    dense_predictor.eval()

    with torch.no_grad():
        image = image.unsqueeze(0).to(device)  # [1, 3, H, W]

        features = backbone(image)
        hc = hyper(features)

        output = dense_predictor(hc)  # [1, num_classes, H, W]

        prediction = torch.argmax(output, dim=1)  # [1, H, W]

    return prediction.squeeze(0).cpu()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Color map for classes (same as your dataset)
CLASS_COLORS = {
    0: (128, 64, 128),   # flat
    1: (70, 70, 70),     # construction
    2: (153, 153, 153),  # object
    3: (107, 142, 35),   # nature
    4: (70, 130, 180),   # sky
    5: (220, 20, 60),    # human
    6: (0, 0, 142),      # vehicle
    7: (0, 0, 0)         # void
}

def mask_to_color(mask):
    """Convert class index mask → RGB image"""
    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)

    for cls, color in CLASS_COLORS.items():
        color_mask[mask == cls] = color

    return color_mask


def visualize_prediction(image, mask, prediction):
    # Unnormalize image
    image = image.permute(1, 2, 0).cpu().numpy()
    image = (image * 0.5) + 0.5
    image = np.clip(image, 0, 1)

    mask_color = mask_to_color(mask.cpu().numpy())
    pred_color = mask_to_color(prediction.numpy())

    plt.figure(figsize=(12, 4))

    # Input image
    plt.subplot(1, 3, 1)
    plt.imshow(image)
    plt.title("Input Image")
    plt.axis("off")

    # Ground truth
    plt.subplot(1, 3, 2)
    plt.imshow(mask_color)
    plt.title("Ground Truth")
    plt.axis("off")

    # Prediction
    plt.subplot(1, 3, 3)
    plt.imshow(pred_color)
    plt.title("Prediction")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Take one sample from validation set
image, mask = val_dataset[0]

prediction = predict_full_image(image, backbone, hyper, dense_predictor)

visualize_prediction(image, mask, prediction)